# U02 SQL（一）：關聯模型、建表約束與單表查詢

**資料庫管理**・09/17　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

先看懂**資料的數學形狀**（關聯模型），然後學會 SQL 的一半：**定義資料（DDL）**・**維護資料（DML）**・**單表查詢全套**

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 與專題的關係 |
|---|---|---|---|
| 第 0 節 | 15 | **關聯模型**：relation 是笛卡兒積的子集・key・NULL | 一切的數學地基 |
| 第 1 節 | 45 | 型別與 affinity、`CREATE TABLE` 與六種約束、主鍵、`INSERT/UPDATE/DELETE`、UPSERT、generated column、`ALTER`、讀懂錯誤訊息 | 你的應用 schema 全靠這節 |
| 第 2 節 | 40 | `SELECT` 全套：WHERE、**NULL 三值邏輯**、排序切頁、字串／**日期時間**／條件函數、聚合、抽樣 | 報表與日常操作查詢 |
| 實作 | 35 | 單表查詢第 1–18 題：核心 12 題＋選做／加碼 6 題 | 之後你天天寫這些 |

本單元起使用課程範例資料庫 **univ.db（大學選課）**——先跑下一格建好它。

> 135 分鐘主線仍以核心內容與 Lab 第 1–12 題為準；標成「選做／加碼」的延伸格可依現場進度取捨，也適合課後自行探索。

In [ ]:
#@title 📦 資料準備：建立課程範例資料庫 univ.db（先跑它，別急著讀懂——本單元結束你就全看得懂）
import sqlite3, os, pandas as pd

if os.path.exists("univ.db"):
    os.remove("univ.db")
con = sqlite3.connect("univ.db")
con.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE student(
  sid   TEXT PRIMARY KEY,          -- 學號
  name  TEXT NOT NULL,             -- 姓名
  dept  TEXT NOT NULL,             -- 系所
  year  INTEGER CHECK(year BETWEEN 1 AND 4)   -- 年級
);
CREATE TABLE instructor(
  iid   TEXT PRIMARY KEY,
  name  TEXT NOT NULL,
  dept  TEXT NOT NULL,
  salary REAL
);
CREATE TABLE course(
  cid     TEXT PRIMARY KEY,
  title   TEXT NOT NULL,
  dept    TEXT NOT NULL,
  credits INTEGER NOT NULL DEFAULT 3
);
CREATE TABLE takes(                -- 修課紀錄
  sid TEXT REFERENCES student(sid),
  cid TEXT REFERENCES course(cid),
  semester TEXT,                   -- 學期，如 114-1
  grade REAL,                      -- NULL = 在修中
  PRIMARY KEY (sid, cid, semester)
);
CREATE TABLE teaches(              -- 授課紀錄
  iid TEXT REFERENCES instructor(iid),
  cid TEXT REFERENCES course(cid),
  semester TEXT,
  PRIMARY KEY (iid, cid, semester)
);
""")
con.executemany("INSERT INTO student VALUES (?,?,?,?)", [
 ("S001","林佳蓉","統計",3),("S002","陳威廷","統計",3),("S003","張雅筑","統計",3),
 ("S004","李承翰","統計",2),("S005","王思穎","統計",4),("S006","黃冠宇","統計",3),
 ("S007","吳孟軒","資訊",3),("S008","劉子涵","資訊",2),("S009","蔡明修","資訊",4),
 ("S010","許芷瑄","數學",3),("S011","鄭宇翔","數學",2),("S012","謝欣妤","數學",4),
 ("S013","洪偉倫","企管",3),("S014","郭品妍","企管",2),("S015","曾柏勳","企管",3),
 ("S016","賴韻如","統計",1),("S017","周家豪","資訊",1),("S018","江美慧","統計",4),
 ("S019","趙國彬","數學",3),("S020","方語彤","企管",4),
])
con.executemany("INSERT INTO instructor VALUES (?,?,?,?)", [
 ("I01","王教授","統計",118000.0),("I02","李教授","統計",102000.0),
 ("I03","張教授","資訊",111000.0),("I04","陳教授","數學",96000.0),
 ("I05","林教授","企管",99000.0),("I06","徐教授","統計",None),
])
con.executemany("INSERT INTO course VALUES (?,?,?,?)", [
 ("C101","統計學（一）","統計",3),("C102","迴歸分析","統計",3),
 ("C103","資料庫管理","統計",3),("C104","機率論","統計",3),
 ("C201","微積分","數學",4),("C202","線性代數","數學",3),
 ("C301","程式設計","資訊",3),("C302","資料結構","資訊",3),
])
con.executemany("INSERT INTO takes VALUES (?,?,?,?)", [
 ("S001","C101","114-1",88),("S001","C104","114-1",92),("S001","C102","114-2",85),
 ("S001","C103","115-1",None),("S002","C101","114-1",76),("S002","C102","114-2",81),
 ("S002","C103","115-1",None),("S003","C101","114-1",95),("S003","C104","114-1",89),
 ("S003","C102","114-2",91),("S003","C103","115-1",None),("S004","C101","114-2",67),
 ("S004","C201","114-2",72),("S004","C104","115-1",None),("S005","C101","113-1",82),
 ("S005","C102","113-2",78),("S005","C103","114-1",90),("S006","C101","114-1",58),
 ("S006","C104","114-1",61),("S006","C103","115-1",None),("S007","C301","114-1",93),
 ("S007","C302","114-2",87),("S007","C103","115-1",None),("S008","C301","114-2",74),
 ("S008","C302","115-1",None),("S009","C301","113-1",85),("S009","C302","113-2",80),
 ("S009","C202","114-1",77),("S010","C201","114-1",90),("S010","C202","114-2",94),
 ("S011","C201","114-2",63),("S011","C202","115-1",None),("S012","C201","113-1",71),
 ("S012","C202","113-2",75),("S012","C101","114-1",79),("S013","C101","114-1",70),
 ("S013","C103","115-1",None),("S014","C101","114-2",55),("S015","C101","114-1",83),
 ("S015","C102","114-2",88),("S016","C101","115-1",None),("S017","C301","115-1",None),
 ("S018","C101","113-1",96),("S018","C102","113-2",93),("S018","C104","114-1",98),
 ("S018","C103","114-1",94),("S019","C201","114-1",84),("S019","C202","114-2",86),
 ("S020","C101","113-2",73),("S020","C102","114-1",69),
])
con.executemany("INSERT INTO teaches VALUES (?,?,?)", [
 ("I01","C101","114-1"),("I01","C101","114-2"),("I01","C104","114-1"),
 ("I02","C102","114-2"),("I02","C102","114-1"),("I06","C103","115-1"),
 ("I06","C103","114-1"),("I03","C301","114-1"),("I03","C302","114-2"),
 ("I04","C201","114-1"),("I04","C202","114-2"),("I05","C101","113-2"),
])
con.commit()

def q(sql, params=()):
    """跑一句 SELECT，回傳 pandas DataFrame（Colab 會漂亮顯示）"""
    return pd.read_sql_query(sql, con, params=params)

for t in ["student","instructor","course","takes","teaches"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<12}{n:>4} 列")
print("univ.db 就緒 ✅")

### univ.db 綱要（這學期的老朋友）

```
student(sid PK, name, dept, year)          instructor(iid PK, name, dept, salary)
course(cid PK, title, dept, credits)       teaches(iid→instructor, cid→course, semester)
takes(sid→student, cid→course, semester, grade)      ← grade 為 NULL 表示在修中
```

「→」是 **foreign key**：`takes.sid` 的值必須存在於 `student.sid`。今天先專心玩單表，下個單元把表接起來。

# 第 0 節：關聯模型——資料的數學形狀

## 0.1 Codd 的天才提案：所有資料都是「表」

用統計系熟悉的語言：一個 **relation** 就是笛卡兒積的一個有限子集

$$r \subseteq D_1 \times D_2 \times \cdots \times D_n$$

其中 $D_i$ 是第 $i$ 個屬性（attribute／欄位）的值域（domain）。每個元素是一個 **tuple**（列）。

三個立刻要記的心智設定（跟 DataFrame 最大的差異）：

1. **schema vs instance**：表的「格式」vs 某一刻的「內容」——設計 schema 是 U04 的主題。
2. 理論上**列沒有順序、不重複**（集合！）；要順序，查詢時自己 `ORDER BY`。
3. 「relational」指的是 relation（表）這個數學物件，**不是**「表跟表有關係」——常見誤解。

術語：欄數叫 **arity**（度數），列數叫 **cardinality**（基數）。定義不是拿來背的——下一格直接把它「跑」出來。

In [ ]:
# relation ＝ 笛卡兒積的有限子集：把定義跑一次
from itertools import product

dom_sid  = {"S001", "S002", "S003"}          # 學號的值域（示意）
dom_dept = {"統計", "資訊"}                   # 系所的值域
dom_year = {3, 4}                             # 年級的值域

universe = set(product(dom_sid, dom_dept, dom_year))            # D1 × D2 × D3：所有「可能」的列
r = {("S001", "統計", 3), ("S002", "統計", 4), ("S003", "資訊", 3)}   # 某一刻的 instance

print(f"|dom_sid × dom_dept × dom_year| = {len(universe)}（3×2×2 種可能）")
print("r ⊆ 笛卡兒積？", r <= universe)
print("arity（欄數）= 3、cardinality（列數）=", len(r))
print("集合沒有順序：",
      {("S001","統計",3), ("S002","統計",4)} == {("S002","統計",4), ("S001","統計",3)})

In [ ]:
# 查詢＝集合運算：選擇 σ（挑列）與投影 π（挑欄）——SQL 的數學前身（U08 見完整關聯代數）
sel = {t for t in r if t[1] == "統計"}          # σ_[dept='統計'](r)   → 之後的 WHERE
proj = {(t[1],) for t in r}                     # π_[dept](r)          → 之後的 SELECT dept
print("σ dept='統計' →", sel)
print("π (dept)      →", proj, "← 只剩 2 個元素！")
print("因為 relation 是「集合」，投影後重複自動消失——這就是之後 SELECT DISTINCT 的數學原意。")

## 0.2 Key：一列資料的身分證

| 名詞 | 定義 | 例（student） |
|---|---|---|
| superkey | 能唯一識別一列的欄位集合 | {sid}、{sid, name}、全欄位 |
| candidate key | **極小**的 superkey（拿掉任一欄就不唯一） | {sid}；若身分證欄存在也是 |
| **primary key** | 從 candidate key 中**選定**的那一個 | sid |
| **foreign key** | 引用其他表主鍵的欄位（表間的黏著劑） | takes.sid → student.sid |

小練習：`takes(sid, cid, semester, grade)`（誰、哪門課、哪學期、幾分）的合理主鍵是？

<details><summary>答案</summary>

`{sid, cid, semester}` 複合主鍵——同一人同一課同一學期只能有一筆；只用 `{sid, cid}` 就擋掉重修了。**主鍵的選擇是業務規則的宣告**，不是技術細節。
</details>

In [ ]:
# key 是「語意」的宣告，資料只能「否證」它：用 pandas 檢查候選 key 有沒有被資料打臉
df_s = pd.DataFrame([("S001", "林佳蓉", "統計", 3), ("S002", "陳威廷", "統計", 3),
                     ("S003", "張雅筑", "統計", 3), ("S001", "林佳蓉", "資訊", 2)],   # 故意讓 S001 出現兩次！
                    columns=["sid", "name", "dept", "year"])
for cand in [["sid"], ["name"], ["sid", "dept"]]:
    dups = int(df_s.duplicated(subset=cand).sum())
    print(f"{str(cand):16s} 重複 {dups} 筆 →",
          "❌ 被資料否證，不是 key" if dups else "資料沒打臉（注意：這不構成證明！）")
print("\n→ {sid} 被否證了？要嘛這批資料有鬼（重複學號），要嘛你對業務的理解有鬼。")
print("   key 的最終依據是業務規則；宣告成 PRIMARY KEY 之後，資料庫會替你「永遠」守住它（第 1 節）。")

In [ ]:
# 小練習的驗證：takes 的主鍵少一欄會發生什麼事——「重修」被誤判成重複
df_takes = pd.DataFrame([("S001", "C101", "114-1", 58),
                         ("S001", "C101", "115-1", 76),    # 這是重修：合法！
                         ("S001", "C104", "114-1", 92)],
                        columns=["sid", "cid", "semester", "grade"])
print("以 {sid, cid} 當 key      → 重複",
      int(df_takes.duplicated(subset=["sid", "cid"]).sum()), "筆：重修被誤殺（key 太小，擋掉合法資料）")
print("以 {sid, cid, semester} 當 key → 重複",
      int(df_takes.duplicated(subset=["sid", "cid", "semester"]).sum()), "筆：這才是對的主鍵")
print("→ 主鍵選誰＝宣告「什麼情況算同一筆」。選錯的代價是資料進不來、或髒資料進得來。")

In [ ]:
# NULL 初體驗：資料庫的「未知」不是 0、不是空字串——連跟自己比較都不相等
mcon = sqlite3.connect(":memory:")
print("NULL = NULL  →", mcon.execute("SELECT NULL = NULL").fetchone()[0], "（None＝UNKNOWN！）")
print("NULL IS NULL →", mcon.execute("SELECT NULL IS NULL").fetchone()[0], "（要用 IS 才問得到）")
print("1 + NULL     →", mcon.execute("SELECT 1 + NULL").fetchone()[0], "（NULL 會傳染）")
print("→ 統計人請把 NULL 想成 missing data。第 2 節有完整的「NULL 三值邏輯專場」。")

# 第 1 節：定義與維護資料

## 1.1 SQLite 的型別系統：先說清楚它的「怪」

標準 SQL 是**靜態型別**（欄位宣告什麼就只能放什麼）；SQLite 是**動態型別**——值自己帶型別，欄位宣告只是「偏好」（type affinity）。

| 儲存類別 | 放什麼 |
|---|---|
| `NULL` | 未知 |
| `INTEGER` | 64-bit 整數 |
| `REAL` | 浮點數 |
| `TEXT` | UTF-8 字串（中文 OK） |
| `BLOB` | 原始位元組（圖片、檔案） |

沒有獨立的 DATE／BOOLEAN 型別：日期慣例存 `TEXT`（ISO 格式 `'2026-09-17'`，可比較可排序），布林存 0／1。

- 宣告 `INTEGER` 的欄位塞 `'123'` → 自動轉成整數；塞 `'abc'` → **原樣存成文字，不報錯**（下一格親眼看）。
- 課堂慣例：**照標準 SQL 好好宣告型別與約束**——之後換 PostgreSQL 無痛，AI 生成的 SQL 也更可靠。

In [ ]:
# 動態型別現場：typeof() 看每個值真正的儲存型別
con.execute("DROP TABLE IF EXISTS t_demo")
con.execute("CREATE TABLE t_demo(x INTEGER)")
con.executemany("INSERT INTO t_demo VALUES (?)", [(123,), ("456",), ("abc",), (7.5,), (None,)])
con.commit()
q("SELECT x, typeof(x) FROM t_demo")

### affinity 是怎麼判定的？（AI 生出奇怪型別名時你要看得懂）

SQLite 拿「宣告的型別**字串**」比對關鍵字，由上而下第一個中的算數：

| 規則（依序） | 判為 | 例 |
|---|---|---|
| 含 `INT` | INTEGER | `BIGINT`、`INT8` |
| 含 `CHAR`／`CLOB`／`TEXT` | TEXT | `VARCHAR(10)`、`NCHAR` |
| 含 `BLOB` 或沒宣告 | BLOB（原樣存） | `BLOB` |
| 含 `REAL`／`FLOA`／`DOUB` | REAL | `FLOAT`、`DOUBLE` |
| 其他 | NUMERIC（能轉數字就轉） | `STRING`、`DECIMAL` |

冷知識：宣告 `STRING` 會落到 **NUMERIC**（不含任何關鍵字）——`'123'` 存進去會變整數！這是 AI 常踩的坑，下一格證明。

In [ ]:
# affinity 判定實測：五種宣告、同一個值 '123'，存出五種結果
con.execute("DROP TABLE IF EXISTS aff")
con.execute("CREATE TABLE aff(a VARCHAR(10), b FLOAT, c BIGINT, d BLOB, e STRING)")
con.execute("INSERT INTO aff VALUES ('123','123','123','123','123')")
con.commit()
print(q("SELECT a, typeof(a) AS ta, b, typeof(b) AS tb, c, typeof(c) AS tc, "
        "d, typeof(d) AS td, e, typeof(e) AS te FROM aff").to_string(index=False))
print("\n→ VARCHAR→text、FLOAT→real、BIGINT→integer、BLOB→原樣（text）、STRING→integer（！）")
print("   宣告型別請用五個正名：INTEGER / REAL / TEXT / BLOB（＋NUMERIC），別讓 affinity 猜。")

### 【選做／加碼】`'10' > '9'`？比較前到底轉了誰

這題不能只背「SQLite 會自動轉型」。要把三件事拆開：

1. **storage class 屬於值**：每個值當下是 NULL、INTEGER、REAL、TEXT 或 BLOB，可用 `typeof()` 看。
2. **affinity 屬於欄位（以及部分運算式）**：寫入欄位或拿欄位比較時，SQLite 會依 affinity **嘗試**轉換；它不是永久、強制的型別宣告。
3. **比較分兩階段**：先依兩邊的 affinity 規則嘗試轉換，再比較。兩邊都是數值就數值比；兩邊都是 TEXT 就依字串順序比；若儲存類別仍不同，排序是 `NULL < INTEGER/REAL < TEXT < BLOB`。

所以，沒有欄位參與時，兩個字串 literal 都沒有 affinity：`'10' > '9'` 是字串比較，答案是 0；甚至 `10 > '9'` 也不會自動把右邊轉成數字，而是 INTEGER 排在 TEXT 前，所以仍是 0。反過來 `'10' > 9` 卻是 1，只因 TEXT 排在數值後面，**不是**因為它把 `'10'` 當成十。若語意確定是數量，請明寫 `CAST`。

欄位加入後才看得到 affinity 的影響：NUMERIC 欄與 `'9'` 比較，會把可轉換的文字套用 NUMERIC affinity；TEXT 欄與數字 `9` 比較，則會把右邊轉成文字。下一格把四種情況一次跑完。

In [ ]:
# 【選做／加碼】literal 沒有欄位 affinity；先猜四個結果再執行
comparison_cases = [
    ("TEXT vs TEXT", "SELECT '10' > '9'"),
    ("INTEGER vs TEXT", "SELECT 10 > '9'"),
    ("TEXT vs INTEGER", "SELECT '10' > 9"),
    ("explicit CAST", "SELECT CAST('10' AS INTEGER) > CAST('9' AS INTEGER)"),
]
for case_name, sql in comparison_cases:
    result = con.execute(sql).fetchone()[0]
    print(f"{case_name:16s} → {result}")

con.execute("DROP TABLE IF EXISTS affinity_compare")
con.execute("CREATE TABLE affinity_compare(num_value NUMERIC, text_value TEXT)")
con.executemany("INSERT INTO affinity_compare VALUES (?,?)", [("10", "10"), ("9", "9")])
con.commit()
print("\n同樣輸入 '10' 與 '9'，寫入後的 storage class 與比較結果：")
q("""SELECT num_value, typeof(num_value) AS num_type,
            num_value > '9' AS num_gt_9,
            text_value, typeof(text_value) AS text_type,
            text_value > 9 AS text_gt_9
     FROM affinity_compare ORDER BY num_value""")
# 重點：不是「SQLite 永遠把數字字串轉成數字」；有沒有欄位 affinity，答案可能不同。

### 隨堂練習 A（2 分鐘，先猜再跑）

`CREATE TABLE p(x DOUBLE, y CLOB, z INT8)`，塞進 `('9', '9', '9')`——三個 typeof 各是什麼？

<details><summary>驗證程式與答案</summary>

```python
con.execute("DROP TABLE IF EXISTS p")
con.execute("CREATE TABLE p(x DOUBLE, y CLOB, z INT8)")
con.execute("INSERT INTO p VALUES ('9','9','9')")
print(q("SELECT typeof(x), typeof(y), typeof(z) FROM p"))
```
`real`、`text`、`integer`——DOUBLE 含 DOUB、CLOB 含 CLOB、INT8 含 INT。規則表就是這樣查的。
</details>

In [ ]:
# 想要「嚴格模式」？SQLite 3.37+ 的 STRICT 表：型別不對直接報錯
if sqlite3.sqlite_version_info >= (3, 37):
    con.execute("DROP TABLE IF EXISTS strict_demo")
    con.execute("CREATE TABLE strict_demo(x INTEGER) STRICT")
    try:
        con.execute("INSERT INTO strict_demo VALUES ('abc')")
        print("⚠️ 竟然過了？！")
    except sqlite3.IntegrityError as e:
        print("✅ STRICT 表擋下 →", e)
    con.commit()
else:
    print("這顆 SQLite <3.37 沒有 STRICT 表——知道有這功能即可")
# 本課不強制 STRICT，知道就好；好好宣告型別＋約束已經夠安全

## 1.2 `CREATE TABLE`：把「資料的規則」宣告出來

```sql
CREATE TABLE registration(                    -- 以社團活動報名為例
  reg_id   INTEGER PRIMARY KEY,               -- 自動編號（見 1.3）
  member_id INTEGER NOT NULL REFERENCES member(member_id),
  event_id  INTEGER NOT NULL REFERENCES event(event_id),
  status    TEXT NOT NULL DEFAULT '報名'
            CHECK (status IN ('報名','候補','取消')),
  reg_time  TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),
  UNIQUE (member_id, event_id)                -- 同人同活動只能報一次
);
```

| 約束 | 保證 | 沒有它的慘案 |
|---|---|---|
| `PRIMARY KEY` | 唯一＋非空，一列一身分 | 同學號兩筆，成績算誰的？ |
| `NOT NULL` | 必填 | 沒姓名的學生 |
| `UNIQUE` | 不重複（可多欄組合） | 同人重複報名 |
| `CHECK (…)` | 值域／商業規則 | 年級 17、名額 −3 |
| `DEFAULT` | 預設值（可以是運算式） | 每筆都要手填時間 |
| `FOREIGN KEY` | 參照完整性 | 幽靈學生的報名 |

⚠️ **SQLite 陷阱**：FK 檢查預設是**關**的！每條連線都要 `PRAGMA foreign_keys = ON;`（setup 已開；你的專題第一格也要開）。

In [ ]:
# 約束是「自動守門員」：一條一條踩給你看
con.execute("DROP TABLE IF EXISTS club")
con.execute("""
CREATE TABLE club(
  club_id  INTEGER PRIMARY KEY,
  cname    TEXT NOT NULL UNIQUE,
  captain  TEXT NOT NULL REFERENCES student(sid),
  fee      INTEGER DEFAULT 0 CHECK (fee >= 0)
)""")
con.execute("INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S001')")

violations = [
    ("NOT NULL", "INSERT INTO club(cname, captain) VALUES (NULL, 'S002')"),
    ("UNIQUE",   "INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S003')"),
    ("CHECK",    "INSERT INTO club(cname, captain, fee) VALUES ('登山社', 'S004', -100)"),
    ("FK",       "INSERT INTO club(cname, captain) VALUES ('吉他社', 'S999')"),   # 幽靈學生
]
for label, sql in violations:
    try:
        con.execute(sql)
        print(f"⚠️ {label}：竟然過了？！")
    except sqlite3.IntegrityError as e:
        print(f"✅ {label:8s} 擋下 → {e}")
con.commit()
q("SELECT * FROM club")   # 資料乾乾淨淨，只有合法的那一筆

> **📌 專題連結**：翻開你的題目找「必備功能」，幾乎每一條都是約束：
> 報名系統的名額 `CHECK(quota >= 0)`、訂票系統的同座位 `UNIQUE(showtime_id, seat_id)`、訂房系統的 `CHECK(check_in < check_out)`、問卷系統的防重複填答 `UNIQUE(survey_id, respondent_id)`⋯⋯
> **能用約束擋的，就不要只靠 Python 的 if**——約束是最後一道防線，多人同時操作時只有它靠得住（U06 會示範為什麼）。

In [ ]:
# DEFAULT 可以是運算式：時間戳自動蓋章（要加括號！AI 常忘）
con.execute("DROP TABLE IF EXISTS notices")
con.execute("""CREATE TABLE notices(
  id      INTEGER PRIMARY KEY,
  body    TEXT NOT NULL,
  created TEXT NOT NULL DEFAULT (datetime('now','+8 hours')))""")
con.execute("INSERT INTO notices(body) VALUES ('週四停課一次')")     # 完全沒提 created
con.execute("INSERT INTO notices(body) VALUES ('教室改 B203')")
con.commit()
q("SELECT * FROM notices")
# 少了括號 DEFAULT datetime('now') 會直接語法錯誤——這是 SQLite 的規定：運算式預設值要括起來

In [ ]:
# CHECK 可以跨欄位：把「起 < 訖」這種規則寫進表裡
con.execute("DROP TABLE IF EXISTS room_use")
con.execute("""CREATE TABLE room_use(
  room TEXT NOT NULL,
  s    TEXT NOT NULL,
  e    TEXT NOT NULL,
  CHECK (s < e))""")                     # 跨欄位規則：起訖顛倒直接擋
con.execute("INSERT INTO room_use VALUES ('圓桌室', '2026-09-17 10:00', '2026-09-17 12:00')")
try:
    con.execute("INSERT INTO room_use VALUES ('圓桌室', '2026-09-17 15:00', '2026-09-17 14:00')")
except sqlite3.IntegrityError as e:
    print("✅ 起訖顛倒被擋 →", e)
con.commit()
q("SELECT * FROM room_use")

### 【選做／加碼】CHECK + GLOB：把編碼格式也寫進 schema

SQLite 核心沒有內建通用的 regular expression；簡單、固定長度的格式可用 GLOB。它是 **glob pattern，不是 regex**：

| 符號 | 意思 |
|---|---|
| <code>*</code> | 任意長度的任意字元 |
| <code>?</code> | 恰好一個任意字元 |
| <code>[0-9]</code> | 恰好一個 ASCII 數字 |

例如 <code>S[0-9]*</code> **不等於**「S 後面全是數字」：它只要求 S 後先有一個數字，後面的星號連英文字也吃，所以 <code>S1xyz</code> 會通過。要驗證「S 加三位數」應完整寫成 <code>S[0-9][0-9][0-9]</code>，沒有星號，長度也一起鎖住。

另一個關鍵：SQLite 的 CHECK 只在結果為 0 時拒絕；結果是 NULL 也算通過。因此必填格式一定要搭配 NOT NULL。GLOB 只能驗證字面形狀；像 <code>2026-99-99</code> 即使符合日期外形，也不是有效日期，語意規則仍要另外處理。

In [ ]:
# 【選做／加碼】先看寬鬆 pattern 的漏洞，再讓 CHECK 擋住格式錯誤
loose_pattern = "S[0-9]*"
exact_pattern = "S[0-9][0-9][0-9]"
samples = ["S001", "S1xyz", "s001", "S01", "S0001", "S12A"]
for sample in samples:
    loose_ok, exact_ok = con.execute(
        "SELECT ? GLOB ?, ? GLOB ?",
        (sample, loose_pattern, sample, exact_pattern),
    ).fetchone()
    print(f"{sample:5s}  loose={loose_ok}  exact={exact_ok}")

con.execute("DROP TABLE IF EXISTS member_code_demo")
con.execute("""CREATE TABLE member_code_demo(
  code TEXT PRIMARY KEY NOT NULL
       CHECK (code GLOB 'S[0-9][0-9][0-9]'))""")
for candidate in ["S001", "S999", "s001", "S01", "S0001", "S12A", "S001x", None]:
    try:
        con.execute("INSERT INTO member_code_demo VALUES (?)", (candidate,))
        print(f"✅ 接受 {candidate!r}")
    except sqlite3.IntegrityError as e:
        print(f"⛔ 擋下 {candidate!r}: {e}")
con.commit()
q("SELECT * FROM member_code_demo ORDER BY code")

In [ ]:
# 複合 UNIQUE：規則是「組合不重複」，單欄可以重複
con.execute("DROP TABLE IF EXISTS seat_pick")
con.execute("""CREATE TABLE seat_pick(
  showtime INTEGER NOT NULL,
  seat     TEXT NOT NULL,
  buyer    TEXT NOT NULL,
  UNIQUE (showtime, seat))""")             # 同場次同座位只能賣一次；不同場次同座位 OK
con.executemany("INSERT INTO seat_pick VALUES (?,?,?)",
                [(1, "A1", "佳蓉"), (2, "A1", "威廷")])       # A1 在兩個場次各賣一次 ✔
try:
    con.execute("INSERT INTO seat_pick VALUES (1, 'A1', '孟軒')")   # 場次 1 的 A1 再賣 → 擋
except sqlite3.IntegrityError as e:
    print("✅ 同場次同座位被擋 →", e)
con.commit()
q("SELECT * FROM seat_pick")
# 「同一 X 的 Y 不可重複」＝ UNIQUE(X, Y)——訂票、預約、報名全是這一句

### FK 的三種「刪除策略」——設計決策，不是技術細節

父列被刪時，指著它的子列怎麼辦？在 `REFERENCES` 後面宣告：

| 寫法 | 行為 | 什麼時候選 |
|---|---|---|
| （不寫）＝ `ON DELETE RESTRICT`* | **擋下刪除**（1.5 示範） | 預設最安全：不准孤兒、也不准連坐 |
| `ON DELETE CASCADE` | 子列**連鎖刪除** | 子列離開父列毫無意義（訂單明細之於訂單） |
| `ON DELETE SET NULL` | 子列的 FK 改成 NULL | 「歸屬」可以懸空（員工的主管離職） |

*嚴格說預設是 NO ACTION，行為與 RESTRICT 幾乎相同。
**心法**：先用預設（擋下），想清楚了才升級 CASCADE——連鎖刪除很方便，誤刪時也很壯烈。

In [ ]:
# CASCADE 現場：刪一篇貼文，它的留言「連坐」蒸發
con.executescript("""
DROP TABLE IF EXISTS comments; DROP TABLE IF EXISTS posts;
CREATE TABLE posts(pid INTEGER PRIMARY KEY, title TEXT NOT NULL);
CREATE TABLE comments(
  cid INTEGER PRIMARY KEY,
  pid INTEGER NOT NULL REFERENCES posts(pid) ON DELETE CASCADE,
  txt TEXT NOT NULL);
INSERT INTO posts(pid, title) VALUES (1, '資料庫好好玩'), (2, '求救：NULL 是什麼');
INSERT INTO comments(pid, txt) VALUES (1, '推'), (1, '先收藏'), (2, '用 IS NULL 啦');
""")
print("刪除前：", q("SELECT COUNT(*) c FROM comments").iloc[0, 0], "則留言")
con.execute("DELETE FROM posts WHERE pid = 1")
con.commit()
print("刪掉貼文 1 之後：")
print(q("SELECT * FROM comments").to_string(index=False))
print("→ 貼文 1 的兩則留言連鎖消失。方便，但也請想像誤刪整個社團時的畫面——所以預設值是「擋下」。")

## 1.3 主鍵的三種常見長相

| 寫法 | 行為 | 用在哪 |
|---|---|---|
| `sid TEXT PRIMARY KEY` | 自然鍵：業務本來就有的唯一編號 | 學號、ISBN、身分證 |
| `id INTEGER PRIMARY KEY` | **代理鍵**：SQLite 幫你自動編號（就是內部 rowid 的別名） | 訂單、報名、貼文⋯⋯大多數表 |
| `PRIMARY KEY (a, b)` | 複合主鍵 | 多對多關聯表（takes、選課） |

`AUTOINCREMENT` 通常**不需要**：`INTEGER PRIMARY KEY` 本來就會遞增編號；`AUTOINCREMENT` 只是多保證「編號永不重用」，還更慢——AI 很愛亂加它，看到可以刪。

In [ ]:
# 代理鍵示範：不給 id，資料庫自己編
con.execute("DROP TABLE IF EXISTS post")
con.execute("CREATE TABLE post(post_id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
for t in ["第一篇", "第二篇", "第三篇"]:
    cur = con.execute("INSERT INTO post(title) VALUES (?)", (t,))
    print(f"插入「{t}」 → 自動編號 post_id = {cur.lastrowid}")
con.commit()
q("SELECT * FROM post")

In [ ]:
# 幕後真相：每張（一般）表都有隱藏的 rowid；INTEGER PRIMARY KEY 就是它的別名
print(q("SELECT rowid, post_id, title FROM post").to_string(index=False))
print("→ rowid 與 post_id 同一個東西（別名）。U07 會看到：整張表就是按 rowid 排的一棵 B-tree。")
print("   刪掉最大列再插入，編號可能重用；要「永不重用」才需要 AUTOINCREMENT（幾乎用不到）。")

## 1.4 `INSERT` 的三種姿勢 ＋ UPSERT

```sql
INSERT INTO club(cname, captain) VALUES ('桌遊社','S002'), ('熱舞社','S003');  -- 一次多列
INSERT INTO takes(sid, cid, semester)                                          -- 從查詢結果塞入
  SELECT sid, 'C103', '115-1' FROM student WHERE dept = '統計' AND year = 4;
INSERT INTO club DEFAULT VALUES;                                               -- 全用預設（少見）
```

**UPSERT**（`ON CONFLICT`）：撞到唯一約束時「改做別的事」而不是報錯——應用系統超常用（重複打卡、重複報名⋯⋯）：

In [ ]:
# UPSERT 之一（DO NOTHING）：打卡系統——同一人同一天重複打卡，不報錯、也不重複
con.execute("DROP TABLE IF EXISTS checkin")
con.execute("""CREATE TABLE checkin(
  sid TEXT REFERENCES student(sid),
  d   TEXT,                        -- 日期
  t   TEXT,                        -- 第一次打卡時間
  PRIMARY KEY (sid, d))""")

def punch_in(sid, d, t):
    con.execute("""INSERT INTO checkin VALUES (?,?,?)
                   ON CONFLICT(sid, d) DO NOTHING""", (sid, d, t))

punch_in("S001", "2026-09-17", "08:55")
punch_in("S001", "2026-09-17", "13:10")   # 同一天第二次 → 靜默忽略，保留最早那筆
punch_in("S002", "2026-09-17", "09:02")
con.commit()
q("SELECT * FROM checkin")

In [ ]:
# UPSERT 之二（DO UPDATE）：頁面點閱計數——第一次插入、之後累加，一句搞定
con.execute("DROP TABLE IF EXISTS page_views")
con.execute("CREATE TABLE page_views(page TEXT PRIMARY KEY, hits INTEGER NOT NULL DEFAULT 1)")

def visit(page):
    con.execute("""INSERT INTO page_views(page) VALUES (?)
                   ON CONFLICT(page) DO UPDATE SET hits = hits + 1""", (page,))

for p in ["首頁", "首頁", "課表", "首頁", "成績", "課表"]:
    visit(p)
con.commit()
q("SELECT * FROM page_views ORDER BY hits DESC")
# 想引用「本來要插入的值」用 excluded.欄名：DO UPDATE SET t = excluded.t（進階，備著）

### `OR IGNORE` 不等於具名目標的 UPSERT `DO NOTHING`

SQLite 有兩套長得像、但**作用範圍不同**的寫法：

| 寫法 | 衝突目標 | 遇到其他壞資料 |
|---|---|---|
| `INSERT OR IGNORE ...` | 沒有 target；是整句 INSERT 的 SQLite conflict policy | UNIQUE／PK／NOT NULL／CHECK 等可適用的違規都可能被**靜默略過**；FK 違規仍會報錯 |
| `... ON CONFLICT(sid, d) DO NOTHING` | 只處理指定的 UNIQUE／PK 衝突 | 其他約束違規照常報錯 |

所以前面打卡例子用 UPSERT：它說清楚「只忽略同人同日」，不會順便吞掉別的資料錯誤。

`INSERT OR REPLACE` 更不是 update：撞到 UNIQUE／PK 時會**刪掉舊列，再插入新列**。因此 rowid 可能變、沒給的欄位回到 DEFAULT，連鎖 FK 還可能把子列一起刪掉。AI 寫出 `OR REPLACE` 時，別把它當成無害的 UPSERT。

In [ ]:
# 親眼比較：OR IGNORE 可吞掉 NOT NULL；指定 target 的 DO NOTHING 不會
con.executescript("""
DROP TABLE IF EXISTS policy_demo;
CREATE TABLE policy_demo(k TEXT PRIMARY KEY, required_text TEXT NOT NULL);
INSERT INTO policy_demo VALUES ('A', 'ok');
""")
con.execute("INSERT OR IGNORE INTO policy_demo VALUES ('B', NULL)")
print("OR IGNORE 後有 B 嗎？", con.execute("SELECT COUNT(*) FROM policy_demo WHERE k='B'").fetchone()[0])
try:
    con.execute("""INSERT INTO policy_demo VALUES ('B', NULL)
                   ON CONFLICT(k) DO NOTHING""")
except sqlite3.IntegrityError as e:
    print("指定 k 的 DO NOTHING 不會吞 NOT NULL 錯誤 →", e)

# OR REPLACE 的「刪＋插」：舊 id 消失、DEFAULT 重設、ON DELETE CASCADE 刪掉子列
con.executescript("""
DROP TABLE IF EXISTS replace_child; DROP TABLE IF EXISTS replace_parent;
CREATE TABLE replace_parent(
  id INTEGER PRIMARY KEY, code TEXT UNIQUE, label TEXT NOT NULL DEFAULT '預設');
CREATE TABLE replace_child(
  id INTEGER PRIMARY KEY, parent_id INTEGER REFERENCES replace_parent(id) ON DELETE CASCADE);
INSERT INTO replace_parent(code, label) VALUES ('A', '舊說明');
INSERT INTO replace_child(parent_id) VALUES (last_insert_rowid());
""")
old_id = con.execute("SELECT id FROM replace_parent WHERE code='A'").fetchone()[0]
con.execute("INSERT OR REPLACE INTO replace_parent(code) VALUES ('A')")
new_row = con.execute("SELECT id, code, label FROM replace_parent WHERE code='A'").fetchone()
child_count = con.execute("SELECT COUNT(*) FROM replace_child").fetchone()[0]
print("OR REPLACE：舊 id =", old_id, "→ 新列 =", new_row, "；剩下子列 =", child_count)
assert new_row[0] != old_id and new_row[2] == '預設' and child_count == 0
con.commit()
print("→ 真正要保留同一列及其子列，請用 ON CONFLICT(...) DO UPDATE。")

In [ ]:
# INSERT ... SELECT：把查詢結果整批搬家（畢業生歸檔）
con.execute("DROP TABLE IF EXISTS alumni")
con.execute("CREATE TABLE alumni(sid TEXT PRIMARY KEY, name TEXT, dept TEXT)")
n = con.execute("INSERT INTO alumni SELECT sid, name, dept FROM student WHERE year = 4").rowcount
con.commit()
print(f"歸檔 {n} 位（期望 5）")
q("SELECT * FROM alumni ORDER BY sid")

In [ ]:
# RETURNING（SQLite 3.35+）：寫入的同時把結果拿回來——省一次查詢
if sqlite3.sqlite_version_info >= (3, 35):
    row = con.execute("INSERT INTO post(title) VALUES ('第四篇') RETURNING post_id, title").fetchone()
    con.commit()
    print("剛插入的列：", row)
else:
    print("這顆 SQLite <3.35 沒有 RETURNING；用 cur.lastrowid 拿自動編號即可")
# 應用場景：新增訂單後立刻要訂單編號去建明細——RETURNING 一趟搞定

## 1.5 `UPDATE`／`DELETE`：鐵律——先想 WHERE

```sql
UPDATE takes SET grade = 61 WHERE sid='S006' AND cid='C101' AND semester='114-1';
DELETE FROM club WHERE cname = '熱舞社';
```

下一格示範忘記 WHERE 的世界末日——以及怎麼靠**交易**逃生（U05 正式教，今天先見識）。

In [ ]:
# 「忘記 WHERE」災難現場（在交易裡演，演完回滾，資料毫髮無傷）
con.execute("BEGIN")
n_hit = con.execute("UPDATE takes SET grade = 100").rowcount     # 少了 WHERE！
print(f"UPDATE takes SET grade = 100  →  改掉了 {n_hit} 列（全班全科 100 分）")
print("平均分數變成：", con.execute("SELECT AVG(grade) FROM takes").fetchone()[0])
con.execute("ROLLBACK")                                          # 交易回滾＝時光倒流
print("ROLLBACK 之後平均：", round(con.execute("SELECT AVG(grade) FROM takes").fetchone()[0], 2))
print("→ 實務守則：改資料前先 BEGIN；先用同條件 SELECT 看會動到哪些列。")

In [ ]:
# UPDATE 的值可以是運算式；rowcount 告訴你「動了幾列」——改完必看的數字
con.execute("BEGIN")
n = con.execute("""UPDATE instructor SET salary = ROUND(salary * 1.05)
                   WHERE dept = '統計' AND salary IS NOT NULL""").rowcount
print(f"統計系調薪 5% → 影響 {n} 列（預期 2：徐教授 salary 是 NULL，不動）")
print(q("SELECT name, salary FROM instructor ORDER BY iid").to_string(index=False))
con.execute("ROLLBACK")     # 課堂示範完回滾；正式要生效改成 con.commit()
print("（已回滾）→ 檢查 rowcount 是應用系統的好習慣：預期改 1 列卻改了 0 或 50 列，就是 bug 的味道。")

In [ ]:
# FK 也擋刪除：有修課紀錄的學生不能直接消失
try:
    con.execute("DELETE FROM student WHERE sid = 'S001'")
except sqlite3.IntegrityError as e:
    print(f"✅ 擋下 → {e}")
    print("   設計選項：預設擋下（RESTRICT）/ ON DELETE CASCADE / SET NULL——1.2 的策略表")
con.commit()

In [ ]:
# 安全刪除三步舞：SELECT 看到 → DELETE 同條件 → rowcount 對帳（跟預覽筆數一樣才安心）
con.execute("INSERT OR IGNORE INTO club(cname, captain) VALUES ('快閃社', 'S002')")
con.commit()
preview = q("SELECT * FROM club WHERE cname = '快閃社'")
print("① 預覽會刪到誰："); print(preview.to_string(index=False))
n = con.execute("DELETE FROM club WHERE cname = '快閃社'").rowcount
con.commit()
print(f"② 執行刪除 → rowcount = {n}")
print("③ 對帳：預覽", len(preview), "列 vs 實刪", n, "列 →", "✅ 一致" if n == len(preview) else "❌ 不對勁！")

### DDL 決策樹（設計時腦中跑一遍）

```
這個欄位…
├─ 一定要有值？ ────────────── NOT NULL
├─ 不能跟別列重複？ ─────────── UNIQUE（組合不重複 → UNIQUE(a, b)）
├─ 有值域／商業規則？ ───────── CHECK（跨欄位也行）
├─ 常常是同一個預設？ ───────── DEFAULT（運算式要括號）
├─ 指向別張表的一列？ ───────── REFERENCES ＋ 想好刪除策略
└─ 可以從其他欄算出來？ ─────── generated column（或乾脆別存）
```

## 1.6 generated column：會自己算的欄位

「訂單明細要不要存 `amount = qty × price`？」存了怕不同步、不存每次都要算——
第三條路：宣告成 **generated column**，資料庫保證它永遠等於公式：

```sql
amount REAL GENERATED ALWAYS AS (qty * unit_price)          -- VIRTUAL：查詢時即算（預設）
amount REAL GENERATED ALWAYS AS (qty * unit_price) STORED   -- STORED：寫入時算好存起來
```

（注意：這跟「成交當下的歷史單價」是兩回事——那個要真的存，U04 會分辨。）

In [ ]:
# generated column 實測：算好給你，而且不准你亂塞
con.execute("DROP TABLE IF EXISTS line_items")
con.execute("""CREATE TABLE line_items(
  item_id    INTEGER PRIMARY KEY,
  qty        INTEGER NOT NULL CHECK (qty > 0),
  unit_price REAL    NOT NULL CHECK (unit_price >= 0),
  amount     REAL    GENERATED ALWAYS AS (qty * unit_price))""")
con.executemany("INSERT INTO line_items(qty, unit_price) VALUES (?,?)", [(2, 45), (1, 120), (5, 30)])
con.commit()
print(q("SELECT * FROM line_items").to_string(index=False))
try:
    con.execute("INSERT INTO line_items(qty, unit_price, amount) VALUES (1, 10, 999)")   # 想造假帳？
except sqlite3.OperationalError as e:
    print("\n✅ 亂塞被擋 →", e)
print("→ 「可推導的值」交給公式管，永遠不會不同步。")

## 1.7 改表、砍表、看目錄

```sql
ALTER TABLE student ADD COLUMN email TEXT;      -- 加欄位
ALTER TABLE student RENAME COLUMN email TO mail;-- 改欄名（3.25+）
ALTER TABLE student DROP COLUMN mail;           -- 砍欄位（3.35+）
ALTER TABLE student RENAME TO student_old;      -- 改表名
DROP TABLE IF EXISTS t_demo;                    -- 砍表（連資料一起，無法復原！）
```

SQLite 的 `ALTER` 只有這幾招（改型別、改約束都不行）——大改要「建新表→搬資料→改名」三步舞，AI 很會寫，你要看得懂它在跳什麼。

**資料庫的自我描述**：schema 本身也存在表裡——`sqlite_master`（U06 會看到它就在檔案第 1 頁）。

In [ ]:
# ALTER 三招連跳
con.execute("ALTER TABLE post ADD COLUMN note TEXT")
print("加欄後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
con.execute("ALTER TABLE post RENAME COLUMN note TO memo")
print("改名後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
if sqlite3.sqlite_version_info >= (3, 35):
    con.execute("ALTER TABLE post DROP COLUMN memo")
    print("砍欄後：", [r[1] for r in con.execute("PRAGMA table_info(post)")])
con.commit()

In [ ]:
# 系統目錄：資料庫「知道自己長什麼樣」
print(q("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name").to_string(index=False))
print()
print("PRAGMA table_info(student) →")
print(q("SELECT cid, name, type, [notnull], pk FROM pragma_table_info('student')").to_string(index=False))

## 1.8 讀懂錯誤訊息：sqlite3 的例外家族

錯誤訊息不是懲罰，是**資料庫在跟你講話**。Python sqlite3 把錯誤分成幾類，先認臉：

| 例外 | 在說什麼 | 常見案例 |
|---|---|---|
| `OperationalError` | **SQL 本身**有問題 | 表／欄不存在、語法打錯、資料庫被鎖 |
| `IntegrityError` | SQL 沒錯，**資料違規** | NOT NULL／UNIQUE／CHECK／FK 被觸發 |
| `ProgrammingError` | 你跟 **API** 的溝通出錯 | `?` 數量對不上、連線已關閉 |

三步讀法：**① 哪一類**（決定往哪找）→ **② 哪個物件**（訊息裡有表名／欄名／約束名）→ **③ 哪條規則**。
問 AI 除錯時，**把整段 traceback 原文貼上**（別只貼「它報錯了」）——訊息裡的物件名就是答案的一半。

In [ ]:
# 錯誤動物園：五種常見錯誤各養一隻，看清楚長相
error_zoo = [
    ("查不存在的表", "SELECT * FROM no_such_table"),
    ("查不存在的欄", "SELECT nickname FROM student"),
    ("SQL 打錯字",   "SELEC * FROM student"),
    ("NOT NULL 違規","INSERT INTO club(cname, captain) VALUES (NULL, 'S001')"),
    ("UNIQUE 違規",  "INSERT INTO club(cname, captain) VALUES ('資料科學社', 'S002')"),
]
for label, sql in error_zoo:
    try:
        con.execute(sql)
        print(f"{label}：過了")
    except sqlite3.Error as e:
        print(f"{label:10s} {type(e).__name__:17s}| {e}")
con.rollback()

In [ ]:
# ProgrammingError 最經典的一隻：? 的數量對不上——九成是「忘了逗號的 tuple」
try:
    con.execute("SELECT * FROM student WHERE sid = ?", ("S001"))    # ("S001") 是字串，不是 tuple！
except sqlite3.ProgrammingError as e:
    print("ProgrammingError →", e)
print()
print("修法：('S001',) ——那個逗號就是 tuple 的靈魂：")
print(con.execute("SELECT sid, name FROM student WHERE sid = ?", ("S001",)).fetchone())
# 訊息裡的「there are 4 supplied」＝字串 'S001' 被拆成 4 個字元。看懂訊息，bug 自己招供。

### 隨堂練習 B（5 分鐘，寫在紙上或下一格）

為**手搖飲料店**的「訂單明細」設計 DDL：
一筆明細屬於某張訂單（`order_id`）、某個品項（`product_id`），有數量（1–20 杯）、單價（≥0）、甜度（限「正常／半糖／微糖／無糖」）。
要求：主鍵、兩個 FK、`CHECK` × 3、合理的 `DEFAULT`。寫完跟隔壁互相挑毛病：**「我可以塞什麼垃圾進你的表？」**

<details><summary>參考解答</summary>

```sql
CREATE TABLE order_item(
  item_id    INTEGER PRIMARY KEY,
  order_id   INTEGER NOT NULL REFERENCES orders(order_id),
  product_id INTEGER NOT NULL REFERENCES product(product_id),
  qty        INTEGER NOT NULL DEFAULT 1 CHECK (qty BETWEEN 1 AND 20),
  unit_price REAL    NOT NULL CHECK (unit_price >= 0),
  sweetness  TEXT    NOT NULL DEFAULT '正常'
             CHECK (sweetness IN ('正常','半糖','微糖','無糖'))
);
```
（也可以加 `UNIQUE(order_id, product_id, sweetness)` 防同單重複列——這是設計判斷，說得出理由就好。
加碼思考：小計欄要用 generated column 嗎？`qty * unit_price` 可以；但 `unit_price` 本身要「存成交當下的價」，別用 FK 即時去撈商品表——U04 會講為什麼。）
</details>

In [ ]:
# 隨堂練習 B 工作區
ex_con = sqlite3.connect(":memory:")

# TODO：CREATE TABLE order_item(...)，然後塞一筆合法、試三筆違規




# 第 2 節：單表查詢全套

> **閱讀路徑**：主線先掌握 `SELECT` 骨架、**§2.2 NULL 三值邏輯**、排序、日期基礎與**月底陷阱**，再進入聚合核心。標成「選讀／工具箱」的格式化、清洗與情境案例可依現場時間跳過，內容保留供實作時查用。

## 2.1 `SELECT` 骨架：書寫順序 ≠ 執行順序

```sql
SELECT DISTINCT 欄位或運算式         -- (5) 挑欄位（投影）
FROM   表                            -- (1) 從哪張表
WHERE  列的條件                      -- (2) 逐列過濾
GROUP BY ...  HAVING ...             -- (3)(4) 分組與組過濾（下個單元）
ORDER BY 排序鍵                      -- (6) 排序
LIMIT  n OFFSET m;                   -- (7) 切前幾筆
```

括號裡是**邏輯執行順序**。記住它，之後學 GROUP BY 和 window functions 都不會迷路——下面立刻看兩個「不懂順序就中招」的現場。

In [ ]:
q("SELECT name, dept, year FROM student WHERE dept = '統計' AND year >= 3")

In [ ]:
# 別名（AS）與執行順序：WHERE(2) 跑在 SELECT(5) 之前——但 SQLite 偷偷幫你
r = q("SELECT sid, grade, grade * 1.05 AS adj FROM takes WHERE adj > 90 ORDER BY adj DESC")
print(r.head(3).to_string(index=False))
print(f"\n共 {len(r)} 列（期望 16）——咦，WHERE 用別名竟然可以？")
print("→ 這是 SQLite 的『寬容方言』：它自動把別名代換回運算式。PostgreSQL / MySQL 會直接報錯！")
print("   可攜寫法：WHERE grade * 1.05 > 90。ORDER BY 在 SELECT 之後，用別名排序則人人合法。")

In [ ]:
# AND 比 OR 黏——不加括號，條件的意思整個歪掉
no_paren   = q("SELECT * FROM takes WHERE cid='C101' OR cid='C104' AND grade >= 90")
with_paren = q("SELECT * FROM takes WHERE (cid='C101' OR cid='C104') AND grade >= 90")
print(f"WHERE cid='C101' OR cid='C104' AND grade>=90 → {len(no_paren)} 列（期望 15）")
print(f"WHERE (cid='C101' OR cid='C104') AND grade>=90 → {len(with_paren)} 列（期望 4）")
print("→ 上句實際是：C101 全部 ∪（C104 且 ≥90）。OR 出沒，必加括號——AI 生的長 WHERE 尤其要檢查。")

In [ ]:
# BETWEEN（含兩端）／ IN ／ 條件組合
q("""SELECT sid, cid, grade FROM takes
     WHERE grade BETWEEN 85 AND 95
       AND cid IN ('C101','C102','C104')
     ORDER BY grade DESC""")

In [ ]:
# LIKE：% 任意長度、_ 恰一個字元（ASCII 不分大小寫；GLOB 用 * ? 且分大小寫）
print(q("SELECT sid, name FROM student WHERE name LIKE '林%'").to_string(index=False))   # 姓林
print()
print(q("SELECT cid, title FROM course WHERE title LIKE '%統計%'").to_string(index=False))  # 課名含統計
print()
print(q("SELECT sid FROM student WHERE sid GLOB 'S0[01]*'").head(6).to_string(index=False)) # GLOB 可用字元集

In [ ]:
# LIKE 的兩個細節：大小寫、以及「找字面上的 %」
print("'s001' LIKE 'S%'  →", con.execute("SELECT 's001' LIKE 'S%'").fetchone()[0], "（ASCII 不分大小寫！學號比對請先統一大小寫或用 GLOB）")
print("'s001' GLOB 'S*'  →", con.execute("SELECT 's001' GLOB 'S*'").fetchone()[0], "（GLOB 分大小寫）")
print("找「50%off」這種含 % 的字面值 → 用 ESCAPE：")
print("'50%off' LIKE '50\\%%' ESCAPE '\\' →",
      con.execute(r"SELECT '50%off' LIKE '50\%%' ESCAPE '\'").fetchone()[0])

## 2.2【主線】NULL 專場：三值邏輯（統計系最常摔的坑）

`grade` 是 NULL 表示「在修中」。直覺會這樣寫：

```sql
SELECT * FROM takes WHERE grade = NULL;     -- 回傳 0 列！
```

因為**任何東西跟 NULL 比較，結果都是 UNKNOWN**（不是 TRUE 也不是 FALSE），而 `WHERE` 只保留 TRUE。

| 規則 | 後果 |
|---|---|
| `x = NULL` → UNKNOWN | 要用 `x IS NULL`／`x IS NOT NULL` |
| NULL 會傳染：`NULL + 1` → NULL | 運算式混進 NULL 整串變 NULL |
| `NOT UNKNOWN` → UNKNOWN | `WHERE NOT (grade > 60)` 一樣撈不到 NULL 列 |
| 聚合函數**忽略** NULL | `AVG(grade)` 只平均有成績的列 |
| `COUNT(*)` 數列、`COUNT(grade)` 數非 NULL | 兩者的差 ＝ NULL 列數 |
| UNIQUE 欄允許**多個 NULL** 共存 | 「未填 email」不會互撞（SQLite／PostgreSQL 行為） |

> 統計對照：NULL ≈ missing data；`AVG` 自動做 listwise deletion——方便，但**你要知道它默默丟了資料**，報表要不要另列「在修中 n 筆」是你的專業判斷。

In [ ]:
print("grade = NULL  →", len(q("SELECT * FROM takes WHERE grade = NULL")), "列（陷阱！）")
print("grade IS NULL →", len(q("SELECT * FROM takes WHERE grade IS NULL")), "列（在修中）")
print("NOT (grade >= 60) →", len(q("SELECT * FROM takes WHERE NOT (grade >= 60)")), "列（NULL 列一樣被排除）")
q("""SELECT COUNT(*)                AS 總列數,
            COUNT(grade)            AS 有成績,
            COUNT(*) - COUNT(grade) AS 在修中,
            ROUND(AVG(grade), 2)    AS 平均_忽略NULL
     FROM takes""")

In [ ]:
# NOT IN 的隱藏地雷：名單裡混進一個 NULL，整句「全軍覆沒」
print("cid NOT IN ('C101','C999')      →", len(q("SELECT * FROM takes WHERE cid NOT IN ('C101','C999')")), f"列（期望 37）")
print("cid NOT IN ('C101', NULL)       →", len(q("SELECT * FROM takes WHERE cid NOT IN ('C101', NULL)")), "列 ← 一列都不剩！")
print()
print("原因：x NOT IN (a, NULL) ≡ x<>a AND x<>NULL，後半永遠 UNKNOWN → 整條過不了 WHERE。")
print("最常中招的場景：NOT IN (SELECT ...) 而子查詢帶回了 NULL。")
print("防法：子查詢加 WHERE 欄 IS NOT NULL；或改用 NOT EXISTS（下個單元教，天生免疫）。")

In [ ]:
# UNIQUE 遇上 NULL：沒填的 email 可以有很多個，填了的不准撞
con.execute("DROP TABLE IF EXISTS contacts")
con.execute("CREATE TABLE contacts(sid TEXT PRIMARY KEY, email TEXT UNIQUE)")
con.executemany("INSERT INTO contacts VALUES (?,?)",
                [("S001", "jr@stat.tw"), ("S002", None), ("S003", None)])   # 兩個 NULL 和平共處
con.commit()
print(q("SELECT * FROM contacts").to_string(index=False))
try:
    con.execute("INSERT INTO contacts VALUES ('S004', 'jr@stat.tw')")
except sqlite3.IntegrityError as e:
    print("\n✅ 重複 email 被擋 →", e)
print("→ NULL 彼此「不相等」所以不算重複。想連沒填都不准：再加 NOT NULL。")

In [ ]:
# NULL 的排序位置可以指定（SQLite 3.30+）：把「在修中」放最後
q("""SELECT sid, cid, grade FROM takes
     WHERE sid IN ('S001','S002')
     ORDER BY grade DESC NULLS LAST""")

## 2.3 排序、去重、切頁

In [ ]:
# 多鍵排序：先系所（升冪），同系再依年級（降冪）；LIMIT+OFFSET 是「分頁」的原型
q("SELECT name, dept, year FROM student ORDER BY dept, year DESC LIMIT 8 OFFSET 0")

In [ ]:
q("SELECT DISTINCT dept FROM student ORDER BY dept")    # 有哪些系（去重）

## 2.4 運算式與內建函數

| 類別 | 常用款 |
|---|---|
| 字串 | `a \|\| b` 串接、`upper/lower`、`length`、`substr(s, 起, 長)`、`replace`、`trim`、`instr`、`printf` |
| 數值 | `round(x, 位數)`、`abs`、`min(a,b)`／`max(a,b)`（雙參數版是逐列比！） |
| 條件 | `CASE WHEN … THEN … ELSE … END` |
| 空值 | `COALESCE(x, y, …)` 第一個非 NULL、`NULLIF(a, b)` 相等變 NULL（防除以零神器） |
| 型別 | `CAST(x AS REAL)`——**整數除法陷阱**的解藥（下面示範） |
| 日期 | 下一小節專場 |

In [ ]:
# 字串函數＋整數除法陷阱（統計人算比例必踩）
print(q("SELECT sid || '－' || name AS 名牌, length(name) AS 字數 FROM student LIMIT 3").to_string(index=False))
print()
print("37 / 50   =", con.execute("SELECT 37 / 50").fetchone()[0], "  ← 整數除整數：小數被截掉！")
print("37 * 1.0 / 50 =", con.execute("SELECT 37 * 1.0 / 50").fetchone()[0])
print("CAST 寫法     =", con.execute("SELECT CAST(37 AS REAL) / 50").fetchone()[0])

In [ ]:
# 【選讀／工具箱】printf／substr 實戰：報表美化與遮罩（個資欄位顯示一半是應用系統日常）
print(q("""SELECT name,
             printf('NT$ %,d', CAST(salary AS INTEGER)) AS 月薪
          FROM instructor WHERE salary IS NOT NULL""").to_string(index=False))
print()
print(q("""SELECT name, substr(sid, 1, 2) || '**' AS 學號遮罩 FROM student LIMIT 3""").to_string(index=False))
# printf 的 %,d 千分位、%05d 補零、%.1f 小數——排版在 SQL 就能做掉一半

In [ ]:
# 【選讀／工具箱】字串清理實戰：把亂七八糟的電話格式洗乾淨（replace 連環拳——匯入舊資料的日常）
con.execute("DROP TABLE IF EXISTS phone_raw")
con.execute("CREATE TABLE phone_raw(who TEXT, phone TEXT)")
con.executemany("INSERT INTO phone_raw VALUES (?,?)", [
    ("佳蓉", "0912-345-678"), ("威廷", "0912 345 679"), ("雅筑", "(09)1234-5680"), ("孟軒", "0912345681")])
con.commit()
q("""SELECT who, phone,
        replace(replace(replace(replace(phone, '-', ''), ' ', ''), '(', ''), ')', '') AS cleaned,
        length(replace(replace(replace(replace(phone, '-', ''), ' ', ''), '(', ''), ')', '')) AS len_ok
     FROM phone_raw""")
# 洗完都是 10 碼 → 之後才能設 UNIQUE、才能比對。心法：先清洗、再約束——順序反了會被自己的髒資料卡死

## 2.5 日期時間專場（應用系統的核心技能）

SQLite 慣例：日期存 `TEXT`（ISO-8601：`'2026-09-17'`／`'2026-09-17 14:30:00'`）——**字串比較順序恰好就是時間順序**，所以能排序能比大小。

| 需求 | 寫法 |
|---|---|
| 台灣現在（本課單時區範例） | `datetime('now','+8 hours')`；今天 `date('now','+8 hours')` |
| 日期運算 | `date('now', '+8 hours', '+7 day')`、`date('now', '+8 hours', '-1 month')` |
| 對齊 | `date('now', '+8 hours', 'start of month')`、`date('now', '+8 hours', 'weekday 1')`（下個週一） |
| 取欄位／分組鍵 | `strftime('%Y-%m', d)` 年月、`'%w'` 星期幾（0=日）、`'%H'` 小時 |
| 差幾天 | `julianday(d2) - julianday(d1)` |

你的專題全都用得到：預約起迄、逾期天數、會籍效期、日結、報名截止⋯⋯

> **Colab 時區地雷**：SQLite 的 `'localtime'` 是「執行程式那台主機的當地時區」，不是使用者瀏覽器的時區；Colab host 通常是 UTC，所以 `localtime` 不會自動變台灣時間。本課全員在台灣的範例明寫 `'+8 hours'`。正式跨時區系統應在資料庫儲存 UTC（`datetime('now')`），顯示時再由應用層用 `zoneinfo.ZoneInfo`（如 `Asia/Taipei`）轉換；不要把所有使用者都假設在 UTC+8。

In [ ]:
# 日期工具箱實測
date_demos = [
    ("今天",            "SELECT date('2026-09-17')"),
    ("7 天後（報名截止）","SELECT date('2026-09-17', '+7 day')"),
    ("本月第一天",       "SELECT date('2026-09-17', 'start of month')"),
    ("這是星期幾(0=日)", "SELECT strftime('%w', '2026-09-17')"),
    ("年月分組鍵",       "SELECT strftime('%Y-%m', '2026-09-17')"),
    ("跨年夜差幾天",     "SELECT julianday('2026-12-31') - julianday('2026-09-17')"),
]
for desc, sql in date_demos:
    print(f"{desc:12s} {sql[7:]:48s} → {con.execute(sql).fetchone()[0]}")

In [ ]:
# 【主線】月底陷阱：1/31 加一個月是幾月幾號？——日期運算的「正規化」行為要親眼看過
for desc, sql in [
    ("1/31 ＋ 1 個月",      "SELECT date('2026-01-31', '+1 month')"),               # 2 月沒有 31 → 溢到 3 月！
    ("本月月底的正解",       "SELECT date('2026-09-17', 'start of month', '+1 month', '-1 day')"),
    ("下月月底",            "SELECT date('2026-01-31', 'start of month', '+2 month', '-1 day')"),
]:
    print(f"{desc:14s} → {con.execute(sql).fetchone()[0]}")
print("\n→ '+1 month' 是「月份加一再正規化」：2026-02-31 不存在 → 自動變 2026-03-03。")
print("   月結、會籍到期這種「每月 X 日」的規則，一律用 start of month 錨定再位移。")

In [ ]:
# 【選讀／工具箱】年齡與生日：日數換算只是近似；精確足歲要比「月-日」
# SQLite 的比較式結果是 0／1；精確足歲在生日尚未到時直接扣 1，原理與可攜寫法見 §2.6。
con.execute("DROP TABLE IF EXISTS bday")
con.execute("CREATE TABLE bday(name TEXT, birth TEXT)")
con.executemany("INSERT INTO bday VALUES (?,?)", [
    ("林佳蓉", "2005-03-14"), ("陳威廷", "2004-11-30"), ("吳孟軒", "2005-09-17")])
con.commit()
q("""SELECT name, birth,
        CAST((julianday('2026-09-17') - julianday(birth)) / 365.2425 AS INTEGER) AS 近似歲數,
        CAST(strftime('%Y','2026-09-17') AS INTEGER)
          - CAST(strftime('%Y', birth) AS INTEGER)
          - (strftime('%m-%d','2026-09-17') < strftime('%m-%d', birth)) AS 精確足歲,
        CASE WHEN strftime('%m-%d', birth) = '09-17' THEN '🎂 今天生日！'
             WHEN strftime('%m-%d', birth) >  '09-17' THEN '今年還沒過'
             ELSE '今年過了' END AS 生日狀態
     FROM bday""")

In [ ]:
# 應用場景演練：借閱到期與逾期天數（教師示範專題「圖書館系統」的核心查詢原型）
con.execute("DROP TABLE IF EXISTS loan_demo")
con.execute("CREATE TABLE loan_demo(who TEXT, book TEXT, due TEXT)")
con.executemany("INSERT INTO loan_demo VALUES (?,?,?)", [
    ("S001", "統計學習導論", "2026-09-10"),   # 已逾期
    ("S002", "資料庫概論",   "2026-09-20"),   # 快到期
    ("S003", "迴歸分析",     "2026-10-05"),   # 還早
])
con.commit()
q("""SELECT who, book, due,
        CAST(julianday('2026-09-17') - julianday(due) AS INTEGER) AS 逾期天數,
        CASE WHEN due <  '2026-09-17' THEN '⚠️ 逾期'
             WHEN due <= date('2026-09-17', '+3 day') THEN '快到期'
             ELSE 'OK' END AS 狀態
     FROM loan_demo ORDER BY due""")

In [ ]:
# CASE WHEN：成績 → 等第（資料重編碼 recode，統計日常）
q("""SELECT sid, cid, grade,
        CASE WHEN grade IS NULL THEN '在修'
             WHEN grade >= 90   THEN 'A'
             WHEN grade >= 80   THEN 'B'
             WHEN grade >= 70   THEN 'C'
             WHEN grade >= 60   THEN 'D'
             ELSE 'F' END AS 等第
     FROM takes ORDER BY grade DESC LIMIT 10""")

In [ ]:
# COALESCE：顯示時把 NULL 換成人話（別改原始資料）；NULLIF：防除以零
print(q("SELECT sid, cid, COALESCE(CAST(grade AS TEXT), '（修課中）') AS 成績 FROM takes LIMIT 6")
        .to_string(index=False))
print()
print("除以零防護：SELECT 10.0 / NULLIF(0, 0) →",
      con.execute("SELECT 10.0 / NULLIF(0, 0)").fetchone()[0], "（NULL，而不是爆炸）")

## 2.6 無分組聚合：整張表濃縮成一列

`COUNT / SUM / AVG / MIN / MAX`——先會這五個；下個單元配 `GROUP BY` 才是完全體。
一個超好用的統計技巧：**`AVG(CASE WHEN 條件 THEN 1.0 ELSE 0 END)` ＝ 條件成立的比例**。

SQLite 的比較結果可當作 `1`／`0`，所以 `SUM(grade < 60)` 能直接數不及格；遇到 NULL 則產生 NULL，`SUM` 會忽略它。這是 **SQLite 慣用簡寫，不是可攜 SQL**：PostgreSQL 等系統請用 `SUM(CASE WHEN … THEN 1 ELSE 0 END)`，或 `COUNT(*) FILTER (WHERE …)`。

In [ ]:
# SQLite 布林簡寫 vs 可攜 CASE：答案應該一樣
short_count = con.execute("SELECT SUM(grade < 60) FROM takes").fetchone()[0]
portable_count = con.execute("""SELECT SUM(CASE WHEN grade < 60 THEN 1 ELSE 0 END)
                                FROM takes""").fetchone()[0]
print("SUM(條件) =", short_count, "；SUM(CASE ...) =", portable_count)
assert short_count == portable_count

In [ ]:
q("""SELECT COUNT(*)                 AS 修課人次,
            COUNT(DISTINCT sid)      AS 修過課的人數,
            ROUND(AVG(grade), 2)     AS 平均,
            MIN(grade) AS 最低, MAX(grade) AS 最高,
            ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3) AS 及格率_含義見下
     FROM takes""")

In [ ]:
# 上面「及格率」分母是誰？NULL 列被 AVG 忽略了嗎？——沒有！CASE 把 NULL 變成 0 了，陷阱！
print("含在修（NULL 當不及格）：",
      con.execute("SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3) FROM takes").fetchone()[0])
print("只算有成績的（正解）　：",
      con.execute("""SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3)
                     FROM takes WHERE grade IS NOT NULL""").fetchone()[0])
print("→ 統計素養時刻：同一句「及格率」，分母定義不同，數字差一截。SQL 沒錯，是你要想清楚。")

In [ ]:
# SUM 的空集合陷阱與 TOTAL：報表右下角的「合計」欄防呆
print("SUM(grade)（沒有任何列中選時）  →", con.execute("SELECT SUM(grade) FROM takes WHERE grade > 200").fetchone()[0])
print("TOTAL(grade)（同條件）          →", con.execute("SELECT TOTAL(grade) FROM takes WHERE grade > 200").fetchone()[0])
print("COUNT(*)（同條件）              →", con.execute("SELECT COUNT(*) FROM takes WHERE grade > 200").fetchone()[0])
print()
print("→ SUM 對空集合回 NULL（會傳染給下游運算）；TOTAL 回 0.0。")
print("   報表寫法二選一：COALESCE(SUM(x), 0) 或 TOTAL(x)——你的 App 合計欄別再顯示 None。")

In [ ]:
# 【選讀／工具箱】聚合不只算數字：group_concat 把一組值串成清單（報表「名單欄」神器）
print("所有系所：", con.execute("SELECT group_concat(DISTINCT dept) FROM student").fetchone()[0])
print("S001 修過：", con.execute("""SELECT group_concat(cid, '、')
                                    FROM (SELECT cid FROM takes
                                          WHERE sid = 'S001' ORDER BY cid)""").fetchone()[0])
print("→ 外層 ORDER BY 只排「聚合完的列」，不保證串接順序；這裡先在子查詢排好。")

In [ ]:
# 【選讀／工具箱】應用：打工班表——「週末時數」佔比（strftime 重編碼 ＋ SUM(CASE) 合體）
con.execute("DROP TABLE IF EXISTS shift")
con.execute("CREATE TABLE shift(who TEXT, d TEXT, hrs REAL)")
con.executemany("INSERT INTO shift VALUES (?,?,?)", [
    ("佳蓉", "2026-09-14", 4), ("佳蓉", "2026-09-19", 6), ("佳蓉", "2026-09-20", 5),
    ("威廷", "2026-09-15", 8), ("威廷", "2026-09-19", 4),
    ("孟軒", "2026-09-16", 6), ("孟軒", "2026-09-17", 6)])
con.commit()
q("""SELECT SUM(hrs) AS 總時數,
        SUM(CASE WHEN strftime('%w', d) IN ('0','6') THEN hrs ELSE 0 END) AS 週末時數,
        ROUND(SUM(CASE WHEN strftime('%w', d) IN ('0','6') THEN hrs ELSE 0 END) * 100.0
              / SUM(hrs), 1) AS 週末佔比pct
     FROM shift""")
# 「每個人」各自的週末時數？那要分堆——下個單元 GROUP BY 一句解決，先欠著

In [ ]:
# 簡單隨機抽樣的 SQL 寫法（做問卷抽獎、抽查都用得到）
q("SELECT sid, cid, grade FROM takes ORDER BY random() LIMIT 5")
# 注意：每次執行結果不同；要可重現的抽樣，之後用 pandas 的 sample(random_state=...) 或預先存亂數欄

### 隨堂練習 C：觀念練習（想好再開）

**Q1.** `WHERE grade <> 100` 會不會回傳 grade 是 NULL 的列？
**Q2.** `COUNT(*)`、`COUNT(grade)`、`COUNT(DISTINCT grade)` 三者的大小關係？
**Q3.** UNIQUE 欄位可以有兩列都是 NULL 嗎？
**Q4.** 你寫 `WHERE cid NOT IN (SELECT ...)`，結果一列都不回。除了「真的沒有」，最可能的原因是？

<details><summary>答案</summary>

**A1.** 不會。`NULL <> 100` 是 UNKNOWN，被 WHERE 濾掉——「不等於」也帶不回 NULL。
**A2.** `COUNT(*) ≥ COUNT(grade) ≥ COUNT(DISTINCT grade)`（NULL 不算入後兩者；重複值再被 DISTINCT 壓縮）。
**A3.** 可以（SQLite／PostgreSQL）：NULL 彼此「不相等」，不算重複。想禁止就再加 `NOT NULL`。
**A4.** 子查詢帶回了 NULL——`x NOT IN (…, NULL)` 整句變 UNKNOWN。防法：子查詢加 `IS NOT NULL`，或改用 `NOT EXISTS`（下個單元）。
</details>

### 隨堂練習 D（3 分鐘）：綜合一句

寫一句 SQL：「所有**有成績**的修課紀錄中，把成績四捨五入到十位（`ROUND(grade, -1)`），
統計每個值出現幾次」——先想：這需要 GROUP BY 嗎？（提示：這題就是要讓你發現需要！）

<details><summary>答案</summary>

```sql
SELECT ROUND(grade, -1) AS bucket, COUNT(*) AS n
FROM takes WHERE grade IS NOT NULL
GROUP BY bucket ORDER BY bucket;
```
對——單表函數玩到極限，自然就撞到「分堆統計」的需求。這正是下個單元 `GROUP BY` 的開場白；
你剛剛畫的其實是一張**直方圖的資料底**。
</details>

## 2.7【AI 協作】本單元示範：用中文描述換一張表

丟給 AI：

> 幫我寫 SQLite 的 CREATE TABLE：社團活動報名表。欄位：報名編號（自動編號主鍵）、
> 學號（必填、參照 student）、活動名稱（必填）、報名時間（預設現在）、
> 繳費金額（不可為負）。同一學號同一活動只能報名一次。

**驗收 SOP**（每次都做）：
1. 跑得過嗎？（貼進 Colab 執行）
2. 每條規則踩一腳：重複報名、負金額、幽靈學號——應該**全部**被擋（寫成 try/except 測試，像 1.2 那格）
3. 追問 AI 兩題，它答得出來、你也要答得出來：
   - 「`AUTOINCREMENT` 在 SQLite 可以拿掉嗎？為什麼？」
   - 「為什麼 `DEFAULT (datetime('now'))` 要加括號？」

> 專題共同要求第 8 條的「應該失敗的測試」，原型就是這套驗收 SOP。

## 2.8【選做／加碼】VIEW 提前一瞥：替查詢取一個可重用的名字

VIEW（檢視）是存在 schema 裡的 SELECT。它看起來像表，但一般 view **不另存一份查詢結果**；每次讀取時都從基底表重新計算，所以原表一改，view 看到的內容也跟著變。

    CREATE VIEW pending_takes_v AS
    SELECT sid, cid, semester
    FROM takes
    WHERE grade IS NULL;

之後可直接執行 <code>SELECT * FROM pending_takes_v</code>。它適合集中重複的欄位命名與篩選規則，但不能放執行時參數；會變動的條件仍寫在查 view 的 WHERE。SQLite 的 view 預設唯讀，新增或修改資料仍對基底表操作（可寫的 view 要另配 INSTEAD OF trigger，先留到後續單元）。

In [ ]:
# 【選做／加碼】view 是「儲存的查詢」，不是結果快照
con.execute("DROP VIEW IF EXISTS pending_takes_v")
con.execute("""CREATE VIEW pending_takes_v AS
               SELECT sid, cid, semester
               FROM takes
               WHERE grade IS NULL""")
con.commit()

view_sql = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='view' AND name='pending_takes_v'"
).fetchone()[0]
before_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("schema 裡存的是：", view_sql)
print("原本 view 有", before_count, "列")

target = con.execute(
    "SELECT sid, cid, semester FROM takes WHERE grade IS NULL LIMIT 1"
).fetchone()
con.execute("BEGIN")
con.execute(
    "UPDATE takes SET grade=80 WHERE sid=? AND cid=? AND semester=?",
    target,
)
after_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("替一筆在修紀錄填成績後，view 立刻變成", after_count, "列")
con.rollback()
restored_count = con.execute("SELECT COUNT(*) FROM pending_takes_v").fetchone()[0]
print("ROLLBACK 後回到", restored_count, "列")
q("SELECT * FROM pending_takes_v ORDER BY sid, cid LIMIT 5")

# 實作時間（35 分鐘）：單表查詢第 1–18 題

規則：每題一格，寫完跑出結果再開解答；**期望輸出**在註解裡幫你自我核對。
難度階梯：第 1–8 題是基本盤，第 9–12 題是主線綜合；**35 分鐘先完成到第 12 題**。
第 13–16 題是選做應用，第 17–18 題是加碼挑戰，不計入 135 分鐘主線。

In [ ]:
# 第 1 題：統計系全部學生的 name、year，年級大的在前
# 期望：8 列，第一列是 4 年級




In [ ]:
# 第 2 題：學分數是 4 的課程（全部欄位）
# 期望：1 列（微積分）




In [ ]:
# 第 3 題：114-1 學期共有幾筆修課紀錄？（一個數字）
# 期望：18




In [ ]:
# 第 4 題：成績 >= 90 的 (sid, cid, grade)，高分在前
# 期望：11 列，最高 98




In [ ]:
# 第 5 題：目前「在修中」的修課紀錄 (sid, cid, semester)
# 期望：11 列




In [ ]:
# 第 6 題：學生來自哪些系所？（不重複）
# 期望：4 列




In [ ]:
# 第 7 題：課名含「統計」或「機率」的課
# 期望：2 列




In [ ]:
# 第 8 題：成績 70–79（含）的修課紀錄有幾筆？
# 期望：10




In [ ]:
# 第 9 題：列出 takes 的 sid, cid, grade，外加一欄「是否及格」：
#          grade IS NULL → '在修'；>= 60 → '及格'；否則 '不及格'
# 期望：50 列；'不及格' 共 2 筆




In [ ]:
# 第 10 題：一句 SQL 回答——修課紀錄總數、有成績筆數、平均（2 位小數）
# 期望：50, 39, 81.08




In [ ]:
# 第 11 題：只看「有成績」的紀錄，及格率是多少？（3 位小數；用 AVG(CASE...) 技巧）
# 期望：0.949




In [ ]:
# 第 12 題：把學號的數字部分取出來轉成整數（substr + CAST），列出 sid 與該整數，取最大的 3 筆
# 期望：S020→20, S019→19, S018→18
# 變化版（做完的人）：改成列出「學號是偶數」的學生（提示：% 2）




In [ ]:
# 第 13 題（選做）：用 UPSERT 寫「借書證領取登記」——同一人重複登記不報錯、保留第一次時間
#   表：CREATE TABLE card_pickup(sid TEXT PRIMARY KEY, picked_at TEXT)
#   期望：對 S001 登記兩次後，表裡只有一筆、時間是第一次的




In [ ]:
# 第 14 題（選做）：把每門課輸出成一欄名牌：「C101｜統計學（一）｜3 學分」（printf 或 || 皆可）
# 期望：8 列，各一個字串




In [ ]:
# 第 15 題（選做）：只用一句 SELECT 比較：
#   (a) 純文字 '10' > '9'；(b) 兩邊明確 CAST 成 INTEGER 後再比較
# 欄名取為 text_order、numeric_order
# 期望：text_order=0, numeric_order=1




In [ ]:
# 第 16 題（選做）：從 raw_codes 找出「大寫 S 加三位 ASCII 數字」的合法代碼
# 期望：2 列，依序為 S001、S123
con.execute("DROP TABLE IF EXISTS raw_codes")
con.execute("CREATE TABLE raw_codes(code TEXT)")
con.executemany("INSERT INTO raw_codes VALUES (?)",
                [("S001",), ("S01",), ("s002",), ("S123",), ("S12A",), ("S0001",)])
con.commit()

# TODO：用 GLOB 寫 SELECT




In [ ]:
# 第 17 題（加碼挑戰）：建立 graded_takes_v view，只保留有成績的 sid, cid, semester, grade
# 接著查詢 view 的列數；寫成整格重跑也不報錯
# 期望：39




In [ ]:
# 第 18 題（加碼挑戰）：做一列修課儀表板
# 欄位：total_rows、graded_rows、pending_rows、passed_rows、failed_rows
# 提示：COUNT(*)、COUNT(grade)、SUM(CASE WHEN ...)
# 期望：50, 39, 11, 37, 2




<details><summary>📖 參考解答（先自己打完再開；跑得出期望輸出的寫法都算對）</summary>

```sql
-- 1
SELECT name, year FROM student WHERE dept = '統計' ORDER BY year DESC;
-- 2
SELECT * FROM course WHERE credits = 4;
-- 3
SELECT COUNT(*) FROM takes WHERE semester = '114-1';
-- 4
SELECT sid, cid, grade FROM takes WHERE grade >= 90 ORDER BY grade DESC;
-- 5
SELECT sid, cid, semester FROM takes WHERE grade IS NULL;
-- 6
SELECT DISTINCT dept FROM student ORDER BY dept;
-- 7
SELECT * FROM course WHERE title LIKE '%統計%' OR title LIKE '%機率%';
-- 8
SELECT COUNT(*) FROM takes WHERE grade BETWEEN 70 AND 79;
-- 9
SELECT sid, cid, grade,
       CASE WHEN grade IS NULL THEN '在修'
            WHEN grade >= 60   THEN '及格'
            ELSE '不及格' END AS 是否及格
FROM takes;
-- 10
SELECT COUNT(*), COUNT(grade), ROUND(AVG(grade), 2) FROM takes;
-- 11
SELECT ROUND(AVG(CASE WHEN grade >= 60 THEN 1.0 ELSE 0 END), 3)
FROM takes WHERE grade IS NOT NULL;
-- 12
SELECT sid, CAST(substr(sid, 2) AS INTEGER) AS n
FROM student ORDER BY n DESC LIMIT 3;
-- 12 變化版
SELECT sid, name FROM student WHERE CAST(substr(sid, 2) AS INTEGER) % 2 = 0;
-- 13（選做）
DROP TABLE IF EXISTS card_pickup;
CREATE TABLE card_pickup(sid TEXT PRIMARY KEY, picked_at TEXT);
INSERT INTO card_pickup VALUES ('S001', '2026-09-17 09:00:00')
ON CONFLICT(sid) DO NOTHING;
INSERT INTO card_pickup VALUES ('S001', '2026-09-17 10:00:00')
ON CONFLICT(sid) DO NOTHING;
SELECT * FROM card_pickup;
-- 14（選做）
SELECT printf('%s｜%s｜%d 學分', cid, title, credits) AS 名牌 FROM course;
-- 15（選做）
SELECT '10' > '9' AS text_order,
       CAST('10' AS INTEGER) > CAST('9' AS INTEGER) AS numeric_order;
-- 16（選做）
SELECT code FROM raw_codes
WHERE code GLOB 'S[0-9][0-9][0-9]'
ORDER BY code;
-- 17（加碼挑戰）
DROP VIEW IF EXISTS graded_takes_v;
CREATE VIEW graded_takes_v AS
SELECT sid, cid, semester, grade
FROM takes
WHERE grade IS NOT NULL;
SELECT COUNT(*) FROM graded_takes_v;
-- 18（加碼挑戰）
SELECT COUNT(*) AS total_rows,
       COUNT(grade) AS graded_rows,
       COUNT(*) - COUNT(grade) AS pending_rows,
       SUM(CASE WHEN grade >= 60 THEN 1 ELSE 0 END) AS passed_rows,
       SUM(CASE WHEN grade < 60 THEN 1 ELSE 0 END) AS failed_rows
FROM takes;
```
</details>

## 自主練習（非繳交）＆ 專題進度建議

**自主練習**（想多練的人，強烈建議）：自選一個有趣主題（追星、健身、遊戲、股票、社團⋯⋯），
1. 造 **2 張表**：一張有 FK 指向另一張，合計用滿 5 種以上約束；
2. 每張 `INSERT` ≥ 10 筆有意義的資料（讓 NULL 合理出現）；
3. 寫 **4 個「應該失敗」的踩點測試**（try/except，模仿 1.2 那格）；
4. 對它寫 **15 個查詢**：LIKE、BETWEEN／IN、`IS NULL`、多鍵 ORDER BY、CASE WHEN、日期函數、聚合、`AVG(CASE…)` 各至少一題。

**不用繳交**——但這正是 U04 工作坊（幫自己的專題設計 schema）最好的暖身。

**專題進度建議**：題目於下個單元（U03）課後公布。本單元先把 Lab 核心第 1–12 題補完；有餘力再依序做第 13–18 題選做／加碼，並把附錄 cheatsheet 掃一遍。下個單元學完 join 你就讀得懂專題題目裡的每一張報表要求。

# 本單元你應該帶走

1. 關聯模型：資料＝relation（笛卡兒積的子集）；schema vs instance；**key 是業務規則的宣告**、資料只能否證它；NULL＝未知。
2. 約束是**寫進 schema 的商業規則**：PK／FK／NOT NULL／UNIQUE／CHECK／DEFAULT，垃圾在寫入瞬間被擋；FK 的刪除策略是設計決策。
3. `PRAGMA foreign_keys = ON`——SQLite 每條連線都要開，專題第一格就寫。
4. UPSERT 兩型（DO NOTHING／DO UPDATE）與 generated column——應用系統的日常武器。
5. NULL 是三值邏輯：`IS NULL` 才抓得到；聚合忽略它；`CASE` 會把它變 0（及格率陷阱）；`NOT IN` 名單混進 NULL 全軍覆沒；`SUM` 空集合回 NULL（用 COALESCE／TOTAL 防）。
6. `UPDATE/DELETE` 先想 WHERE、改完看 rowcount；改資料前 `BEGIN`，錯了 `ROLLBACK`。
7. 日期存 ISO 字串：`date/strftime/julianday` 三板斧＋**月底要用 start of month 錨定**。
8. 錯誤訊息是朋友：`OperationalError`＝SQL 有問題、`IntegrityError`＝資料違規、`ProgrammingError`＝API 用法錯。

**選做／加碼補充**：比較結果同時受 value storage class 與 column affinity 影響，數量語意用明確 CAST；固定格式可用 CHECK + GLOB（並搭配 NOT NULL）；VIEW 是可重用、會隨基底表更新的儲存查詢。

**下個單元**：把表接起來——join 全家、GROUP BY、子查詢，以及統計系的新武器 window functions；**專題指派於課後公布**。讀物：Silberschatz ch2–3（本單元）、ch4–5（預習）；Ullman ch2、ch6。

---
## 附錄 A：本單元 cheatsheet

```sql
-- DDL
CREATE TABLE t(
  id   INTEGER PRIMARY KEY,                  -- 自動編號（= rowid 別名）
  code TEXT NOT NULL UNIQUE,
  kind TEXT NOT NULL DEFAULT 'A' CHECK (kind IN ('A','B')),
  ts   TEXT NOT NULL DEFAULT (datetime('now','+8 hours')),    -- 本課台灣時間；運算式 DEFAULT 要括號
  fk   INTEGER REFERENCES other(id) ON DELETE CASCADE,        -- 或 SET NULL；不寫=擋下
  amt  REAL GENERATED ALWAYS AS (a * b),     -- 會自己算的欄位
  CHECK (s < e),                             -- 跨欄位規則
  UNIQUE (code, kind)                        -- 複合唯一
);
ALTER TABLE t ADD COLUMN note TEXT;          -- 也有 RENAME COLUMN / DROP COLUMN
DROP TABLE IF EXISTS t;
PRAGMA foreign_keys = ON;                    -- 每條連線必開！
-- 固定格式：GLOB 不是 regex，星號代表任意後綴；精確長度要逐字元寫
CREATE TABLE member(code TEXT NOT NULL CHECK(code GLOB 'S[0-9][0-9][0-9]'));
CREATE VIEW pending_takes_v AS SELECT ... FROM takes WHERE grade IS NULL;
DROP VIEW IF EXISTS pending_takes_v;

-- DML
INSERT INTO t(code) VALUES ('x'), ('y');
INSERT INTO t(code) SELECT ... ;
INSERT INTO t VALUES (...) ON CONFLICT(code) DO NOTHING;             -- UPSERT 忽略
INSERT INTO t VALUES (...) ON CONFLICT(code) DO UPDATE SET n = n+1;  -- UPSERT 改寫
INSERT INTO t(code) VALUES ('z') RETURNING id;                       -- 3.35+
UPDATE t SET kind='B' WHERE id=1;            -- 改完看 rowcount！
DELETE FROM t WHERE id=1;

-- 單表 SELECT
SELECT DISTINCT col, expr AS 別名
FROM t
WHERE a BETWEEN 1 AND 9 AND b IN (...) AND c LIKE '林%' AND d IS NOT NULL
  AND (x = 1 OR x = 2)                       -- OR 出沒必加括號（AND 較黏）
ORDER BY a, b DESC NULLS LAST
LIMIT 10 OFFSET 20;

-- 常用函數
COALESCE(x, 0)  NULLIF(a, b)  CAST(x AS REAL)  round(x,2)  length(s)  s1||s2  substr(s,2)
printf('NT$ %,d', n)  instr(s, '關鍵字')
date('now','+8 hours','+7 day')  date(d,'start of month','+1 month','-1 day')   -- 台灣七天後／月底
strftime('%Y-%m', d)  strftime('%w', d)  julianday(d2) - julianday(d1)
CASE WHEN … THEN … ELSE … END
COUNT(*)  COUNT(col)  COUNT(DISTINCT col)  SUM  AVG  MIN  MAX  TOTAL  group_concat(col,'、')
AVG(CASE WHEN 條件 THEN 1.0 ELSE 0 END)     -- 比例
COALESCE(SUM(x), 0)                          -- 合計防 NULL
```

### 錯誤家族速查
| 例外 | 意思 | 第一反應 |
|---|---|---|
| `OperationalError` | SQL 有問題 | 檢查表名／欄名／拼字 |
| `IntegrityError` | 資料違規 | 看訊息點名哪條約束 |
| `ProgrammingError` | API 用法錯 | 檢查 `?` 與參數 tuple（逗號！） |

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| 第 0 節 關聯模型與 key | §1.3、§2.1–2.3 | §2.1–2.2 |
| 1.1–1.7 DDL、約束、DML | §3.2、§4.4（integrity constraints） | §2.3 |
| 2.1–2.4 SELECT、WHERE、函數 | §3.3–3.4 | §6.1 |
| 2.2 NULL 三值邏輯 | §3.6 | §6.1.6–6.1.7 |
| 2.6 聚合 | §3.7 | §6.4 |

SQLite 官方文件（當字典用）：型別與 affinity https://sqlite.org/datatype3.html ・日期函數 https://sqlite.org/lang_datefunc.html ・UPSERT https://sqlite.org/lang_upsert.html